# Azzaro: Whisper turbo vs small vs large

**Objetivo:** retranscribir el mismo video con los tres modelos y revisar los clips
donde producen textos distintos.

Esta version no usa las transcripciones anteriores. Los clips se forman con timestamps
por palabra y pausas reales, usando el mismo criterio de 3–10 segundos del pipeline.


## 1. Configuracion


In [ ]:
from itertools import combinations
from pathlib import Path
import shutil
import sys

try:
    ROOT = Path.cwd().resolve()
except FileNotFoundError:
    ROOT = Path.home() / "labios-argentos"

while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError(
        "No se encontro la raiz del repo. Abrir labios-argentos y reiniciar el kernel."
    )

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, Video, display

from evaluation.src.whisper_model_comparison import (
    alinear_diferencias,
    construir_casos_por_clips_pipeline,
    descargar_video_yt,
    exportar_clips_revision_webm,
    extraer_palabras,
    resumen_diferencias,
    transcribir_whisper,
)

VIDEO_URL = "https://www.youtube.com/watch?v=a4ggqJZXnQE"
VIDEO_PATH = None

# En la Mac usamos MLX para aprovechar Apple Silicon.
BACKEND = "mlx"
MODELOS = ["turbo", "small", "large"]
MODELOS_MLX = {
    "turbo": "mlx-community/whisper-large-v3-turbo",
    "small": "mlx-community/whisper-small-mlx",
    "large": "mlx-community/whisper-large-v3-mlx",
}

# La corrida desde cero ya fue realizada en la Mac. False reutiliza exclusivamente
# los caches nuevos con timestamps por palabra; si no existen, se generan.
REPROCESAR_DESDE_CERO = False
MODELO_CORTE = "turbo"
COOKIES_FROM_BROWSER = "chrome"

# Seleccion manual realizada luego de revisar todos los clips corregidos.
# Son posiciones 1-based dentro de la lista completa de casos.
CASOS_SELECCIONADOS = [4, 5, 6, 10, 14, 24, 33, 34, 35, 42, 43]

WORKDIR = ROOT / "evaluation" / "outputs" / "azzaro_whisper"
TRANSCRIPTS_DIR = WORKDIR / "transcripts_word_timestamps_v2"
CLIPS_DIR = WORKDIR / "clips_pipeline_webm"
WORKDIR


## 2. Cargar las transcripciones corregidas


In [ ]:
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser().resolve()
else:
    video_path = descargar_video_yt(
        VIDEO_URL,
        WORKDIR / "videos",
        nombre_base="azzaro_racing_caracas",
        cookies_from_browser=COOKIES_FROM_BROWSER,
    )

if REPROCESAR_DESDE_CERO:
    for carpeta_generada in (TRANSCRIPTS_DIR, CLIPS_DIR):
        if carpeta_generada.exists():
            shutil.rmtree(carpeta_generada)

resultados = {}
palabras = {}

for modelo in MODELOS:
    print(f"Cargando o transcribiendo {modelo} con {BACKEND}...")
    resultados[modelo] = transcribir_whisper(
        video_path,
        model_name=modelo,
        model_path=MODELOS_MLX[modelo] if BACKEND == "mlx" else None,
        output_dir=TRANSCRIPTS_DIR,
        backend=BACKEND,
        force=REPROCESAR_DESDE_CERO,
    )
    palabras[modelo] = extraer_palabras(resultados[modelo], modelo)

display(pd.DataFrame([
    {"modelo": modelo, "palabras_con_timestamp": len(palabras[modelo])}
    for modelo in MODELOS
]))


## 3. Comparar sobre los clips corregidos


In [ ]:
resumenes = []
for modelo_a, modelo_b in combinations(MODELOS, 2):
    diferencias = alinear_diferencias(
        palabras[modelo_a],
        palabras[modelo_b],
        contexto=0,
    )
    resumen = resumen_diferencias(
        diferencias,
        palabras[modelo_a],
        palabras[modelo_b],
    )
    resumenes.append({
        "comparacion": f"{modelo_a} vs {modelo_b}",
        "grupos_distintos": resumen["grupos_con_diferencias"],
        f"palabras_{modelo_a}": resumen["palabras_turbo_en_diferencias"],
        f"palabras_{modelo_b}": resumen["palabras_small_en_diferencias"],
    })

display(pd.DataFrame(resumenes))

# Turbo define los limites porque es el modelo usado por defecto en el pipeline.
# Cada texto se recupera despues usando exactamente el intervalo del clip.
casos = construir_casos_por_clips_pipeline(
    palabras,
    modelo_corte=MODELO_CORTE,
)

invalidos = [
    numero for numero in CASOS_SELECCIONADOS
    if numero < 1 or numero > len(casos)
]
if invalidos:
    raise IndexError(f"Casos seleccionados fuera de rango: {invalidos}")

casos_revision = [
    {**casos[numero - 1], "caso_seleccionado": numero}
    for numero in CASOS_SELECCIONADOS
]

print(f"Clips del pipeline donde los tres modelos difieren: {len(casos)}")
print(f"Clips seleccionados manualmente: {len(casos_revision)}")

columnas = [
    "caso_seleccionado",
    "clip_id",
    "start",
    "end",
    "transcripcion_turbo",
    "transcripcion_small",
    "transcripcion_large",
]
display(pd.DataFrame(casos_revision)[columnas])


## 4. Escuchar los casos relevantes

Los clips se regeneran en WebM para que VSCode pueda reproducir video y audio. Cada
transcripción corresponde al clip completo mostrado, no a un tramo vecino.


In [ ]:
clips = exportar_clips_revision_webm(
    video_path,
    casos_revision,
    CLIPS_DIR,
    margen=0.08,
    force=True,
)

for caso in clips:
    display(Markdown(
        f"### Caso {int(caso['caso_seleccionado'])} | "
        f"clip_{int(caso['clip_id']):04d} | "
        f"{caso['clip_start']}s-{caso['clip_end']}s"
    ))

    display(Video(
        filename=caso["clip_path"],
        embed=True,
        mimetype="video/webm",
        html_attributes="controls preload='metadata'",
    ))

    print("turbo:", caso["transcripcion_turbo"])
    print("small:", caso["transcripcion_small"])
    print("large:", caso["transcripcion_large"])
